# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: a88e34b2-5a23-431c-a012-0dcbc6abd6d1
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session a88e34b2-5a23-431c-a012-0dcbc6abd6d1 to get into ready status...
Session a88e34b2-5a23-431c-a012-0dcbc6abd6d1 ha

#### Example: Create a DynamicFrame from a table in the AWS Glue Data Catalog and display its schema


In [2]:
order_items_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='order_items')
products_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_clean', table_name='products')
payments_dyf = glueContext.create_dynamic_frame.from_catalog(database='brazilian_ecommerce_raw', table_name='order_payments')

order_items_dyf.printSchema()
products_dyf.printSchema()

root
|-- order_id: string
|-- order_item_id: long
|-- product_id: string
|-- seller_id: string
|-- shipping_limit_date: string
|-- price: double
|-- freight_value: double

root
|-- product_id: string
|-- product_category_name: string
|-- product_name_length: long
|-- product_description_length: long
|-- product_photos_qty: long
|-- product_weight_g: long
|-- product_length_cm: long
|-- product_height_cm: long
|-- product_width_cm: long
|-- product_category_name_english: string
|-- product_dimension_cm3: long


In [3]:
order_items_dyf = order_items_dyf.apply_mapping([
    ("order_id","string","order_id","string"),
    ("order_item_id","string","order_item_id","string"),
    ("product_id","string","product_id","string"),
    ("seller_id","string","seller_id","string"),
    ("shipping_limit_date","timestamp","shipping_limit_date","timestamp"),
    ("price","double","price","double"),
    ("freight_value","double","freight_value","double")
])

#### Example: Convert the DynamicFrame to a Spark DataFrame and display a sample of the data


In [4]:
# I transform both DynamicFrames into pyspark Dataframes to use pyspark SQl
order_items_df = order_items_dyf.toDF()
products_df = products_dyf.toDF()
payments_df = payments_dyf.toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
order_items_df.createOrReplaceTempView("order_items")
products_df.createOrReplaceTempView("products")

joined_oi_df = spark.sql("""
    SELECT * FROM order_items o1
    JOIN products p1 
    ON o1.product_id=p1.product_id
""")

joined_oi_df = joined_oi_df.limit(1000)

In [6]:
# Step 1: Remove rows whose price column has a non positive value ( price <= 0 )
cleaned_oi_df = joined_oi_df.filter(joined_oi_df["price"] <= 0)

# Step 2: Calculate total price for an item, which is price + freight value (shipping costs)
cleaned_oi_df = cleaned_oi_df.withColumn("total_price", joined_oi_df["price"] + joined_oi_df["freight_value"])

# Step 3: Log inconsistencies between order totals and payments
cleaned_oi_df.createOrReplaceTempView("order_items_table")
payments_df.createOrReplaceTempView("payments_table")

# Step 3.1: Calculate the aggregates
order_items_total = spark.sql("""
    SELECT order_id, SUM(total_price) AS order_total
    FROM order_items_table
    GROUP BY order_id
""")

payments_total = spark.sql("""
    SELECT order_id, SUM(payment_value) AS order_total
    FROM payments_table
    GROUP BY order_id
""")

# Step 3.2: Join the tables
order_items_total.createOrReplaceTempView("order_items_table_total")
payments_total.createOrReplaceTempView("payments_table_total")

joined_total = spark.sql("""
    SELECT * FROM order_items_table_total o1
    JOIN payments_table_total p1 
    ON o1.order_id=p1.order_id
""")

# Step 3.3: Save the inconsistencies
inconsistencies_df = joined_total.filter(joined_total["o1.order_total"] != joined_total["p1.order_total"])

In [10]:
# Convert the pyspark Dataframe back into a DynamicFrame
from awsglue.dynamicframe import DynamicFrame

cleaned_dyf = DynamicFrame.fromDF(cleaned_oi_df, glueContext, "order_items_dyf")
inconsistencies_dyf = DynamicFrame.fromDF(inconsistencies_df, glueContext, "inconsistencies_dyf")

In [8]:
# Store the cleaned dataset as Parquet in S3
s3output = glueContext.getSink(
  path="s3://bucket181rt2/clean/order_items",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_clean", catalogTableName="order_items"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(cleaned_dyf)

In [11]:
# Log the inconsistencies in a different database
s3output = glueContext.getSink(
  path="s3://bucket181rt2/inconsistencies/order_items",
  connection_type="s3",
  updateBehavior="UPDATE_IN_DATABASE",
  partitionKeys=[],
  compression="snappy",
  enableUpdateCatalog=True,
  transformation_ctx="s3output",
)
s3output.setCatalogInfo(
  catalogDatabase="brazilian_ecommerce_inconsistencies", catalogTableName="order_items"
)
s3output.setFormat("glueparquet")
s3output.writeFrame(inconsistencies_dyf)